# Analytic Beam Formalism in validation-sim

## Complete Documentation: From CLI to Simulation

This notebook documents the **analytic beam formalism** used in the `validation-sim` codebase for HERA visibility simulations. It covers:

1. **Beam Classes**: Available analytical beam models (AiryBeam, GaussianBeam, PolyBeam, ZernikeBeam, PerturbedPolyBeam)
2. **CLI Options**: All command-line parameters for specifying beams
3. **Configuration Files**: YAML/CSV formats for per-antenna beam mapping
4. **Code Architecture**: Module flow from CLI → obsparams → simulation
5. **Example Commands**: Ready-to-use CLI invocations

---

## Table of Contents

1. [Overview: Beam Mode Hierarchy](#section-1)
2. [Available Beam Classes](#section-2)
3. [CLI Options Reference](#section-3)
4. [Configuration File Formats](#section-4)
5. [Code Architecture & Call Flow](#section-5)
6. [Example Commands](#section-6)
7. [Output & Log File Locations](#section-7)
8. [Source Code Excerpts](#section-8)

<a id="section-1"></a>
## Section 1: Overview — Beam Mode Hierarchy

The `validation-sim` pipeline supports **three beam modes** with the following precedence:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                         BEAM MODE PRECEDENCE                                 │
└─────────────────────────────────────────────────────────────────────────────┘

    HIGHEST PRIORITY
    ────────────────
    1. Multi-Antenna Analytic Beam Map (--analytic-beam-map-file)
       • YAML file mapping antennas to different analytic beam configs
       • Supports ZernikeBeam, PolyBeam, PerturbedPolyBeam
       • Per-antenna beam coefficients and tilts
                    │
                    ▼
    2. Single Analytic Beam (--analytic-beam-class + params)
       • One analytic beam for all antennas
       • AiryBeam, GaussianBeam, PolyBeam, ZernikeBeam, PerturbedPolyBeam
                    │
                    ▼
    3. Per-Antenna UVBeam Files (--beam-map-csv)
       • CSV mapping antennas to .fits beam files
       • Only works with matvis simulator
                    │
                    ▼
    4. Default UVBeam (beams/NF_HERA_Vivaldi_efield_beam_extrap.fits)
       • Single beam file for all antennas
    ────────────────
    LOWEST PRIORITY
```

### Key Points

| Mode | CLI Option | Config Type | Simulator Compatibility |
|------|-----------|-------------|-------------------------|
| Multi-Antenna Analytic | `--analytic-beam-map-file` | YAML | matvis, fftvis |
| Single Analytic | `--analytic-beam-class` | CLI params | matvis, fftvis |
| Per-Antenna UVBeam | `--beam-map-csv` | CSV | **matvis only** |
| Default UVBeam | (none) | — | matvis, fftvis |

<a id="section-2"></a>
## Section 2: Available Beam Classes

### Beam Class Overview

| Class | Source | Parameters | Use Case |
|-------|--------|------------|----------|
| `AiryBeam` | pyuvdata | `diameter` | Simple diffraction-limited dish |
| `GaussianBeam` | pyuvdata | `sigma` or `diameter` | Gaussian approximation |
| `hera_sim.beams.PolyBeam` | hera_sim | `beam_coeffs`, `ref_freq`, `spectral_index` | Chebyshev polynomial model (Fagnoni19) |
| `hera_sim.beams.ZernikeBeam` | hera_sim | `beam_coeffs`, `ref_freq`, `spectral_index`, `peak_normalized` | Zernike polynomial model (tilts/aberrations) |
| `hera_sim.beams.PerturbedPolyBeam` | hera_sim | PolyBeam params + perturbation params | PolyBeam with mainlobe/sidelobe perturbations |

---

### AiryBeam (pyuvdata)

Classic Airy disk diffraction pattern from a circular aperture.

```python
from pyuvdata.analytic_beam import AiryBeam
beam = AiryBeam(diameter=14.0)  # 14m dish
```

**Parameters:**
- `diameter` (float): Dish diameter in meters. Default: 14.0m for HERA

---

### GaussianBeam (pyuvdata)

Simple Gaussian beam profile.

```python
from pyuvdata.analytic_beam import GaussianBeam
beam = GaussianBeam(sigma=0.15)  # sigma in radians
# OR
beam = GaussianBeam(diameter=14.0)  # derives sigma from diameter
```

**Parameters:**
- `sigma` (float): Beam width in radians
- `diameter` (float): Alternative — dish diameter in meters

---

### PolyBeam (hera_sim)

Azimuthally-symmetric beam based on **Chebyshev polynomials**. This is the primary HERA beam model from [Fagnoni+ 2019](http://reionization.org/wp-content/uploads/2013/03/HERA081_HERA_Primary_Beam_Chebyshev_Apr2020.pdf).

```python
from hera_sim.beams import PolyBeam

# Use preset Fagnoni19 coefficients
beam = PolyBeam.like_fagnoni19()

# Or specify custom coefficients
beam = PolyBeam(
    beam_coeffs=[0.297, -0.448, 0.273, -0.100, ...],
    ref_freq=1.0e8,        # 100 MHz reference
    spectral_index=-0.6975,
    polarized=False,
)
```

**Parameters:**
- `beam_coeffs` (list): Chebyshev polynomial coefficients
- `ref_freq` (float): Reference frequency in Hz (default: 1e8)
- `spectral_index` (float): Frequency scaling power (default: -0.6975)
- `polarized` (bool): Include dipole polarization modulation

**Fagnoni19 Preset Coefficients:**
```python
FAGNONI19_COEFFS = [
    0.29778665, -0.44821433, 0.27338272, -0.10030698,
    -0.01195859, 0.06063853, -0.04593295, 0.0107879,
    0.01390283, -0.01881641, -0.00177106, 0.01265177,
    -0.00568299, -0.00333975, 0.00452368, 0.00151808,
    -0.00593812, 0.00351559,
]
```

---

### ZernikeBeam (hera_sim)

Beam model using **Zernike polynomials**, which naturally describe optical aberrations and beam tilts.

```python
from hera_sim.beams import ZernikeBeam

beam = ZernikeBeam(
    beam_coeffs=[1.0, 0.03, 0.01, 0.0, -0.25, ...],  # Up to 66 coefficients
    ref_freq=1.0e8,
    spectral_index=-0.5,
    peak_normalized=True,
)
```

**Parameters:**
- `beam_coeffs` (list): Zernike polynomial coefficients (up to 66)
- `ref_freq` (float): Reference frequency in Hz
- `spectral_index` (float): Frequency scaling power
- `peak_normalized` (bool): Normalize to 1 at beam center

**Zernike Coefficient Meaning (first few):**
| Index | Mode | Physical Effect |
|-------|------|----------------|
| 0 | Piston | Overall amplitude |
| 1 | X-tilt | Pointing error in X |
| 2 | Y-tilt | Pointing error in Y |
| 3 | Oblique astigmatism | |
| 4 | Defocus | Main lobe width |
| 5 | Vertical astigmatism | |

---

### PerturbedPolyBeam (hera_sim)

Extends PolyBeam with additional perturbations for mainlobe, sidelobes, and beam shape transformations.

```python
from hera_sim.beams import PerturbedPolyBeam

beam = PerturbedPolyBeam(
    # Base PolyBeam parameters
    beam_coeffs=[...],
    ref_freq=1.0e8,
    spectral_index=-0.6975,
    # Perturbation parameters
    mainlobe_width=1.1,  # Stretch mainlobe by 10%
    perturb_coeffs=[0.1, 0.05],  # Sidelobe modulation
    rotation_angle=0.1,  # radians
    shear=[0.05, 0.0],   # Beam shear
)
```

In [ ]:
# ============================================================
# Demonstrate Beam Class Initialization
# ============================================================

from pathlib import Path
import numpy as np
import sys

# Add hera_sim to path
HERA_SIM_PATH = Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/hera_sim/src")
if str(HERA_SIM_PATH) not in sys.path:
    sys.path.insert(0, str(HERA_SIM_PATH))

try:
    from pyuvdata.analytic_beam import AiryBeam, GaussianBeam
    from hera_sim.beams import PolyBeam, ZernikeBeam, PerturbedPolyBeam
    HAVE_BEAMS = True
except ImportError as e:
    print(f"Import error: {e}")
    HAVE_BEAMS = False

if HAVE_BEAMS:
    print("Available Beam Classes:")
    print("=" * 50)
    
    # AiryBeam
    airy = AiryBeam(diameter=14.0)
    print(f"\n1. AiryBeam")
    print(f"   diameter: 14.0 m")
    
    # GaussianBeam
    gauss = GaussianBeam(diameter=14.0)
    print(f"\n2. GaussianBeam")
    print(f"   diameter: 14.0 m")
    
    # PolyBeam with Fagnoni19 preset
    poly = PolyBeam.like_fagnoni19()
    print(f"\n3. PolyBeam (Fagnoni19)")
    print(f"   ref_freq: {poly.ref_freq/1e6:.0f} MHz")
    print(f"   spectral_index: {poly.spectral_index}")
    print(f"   beam_coeffs: {len(poly.beam_coeffs)} coefficients")
    
    # ZernikeBeam
    zernike = ZernikeBeam(
        beam_coeffs=[1.0, 0.03, 0.01, 0.0, -0.25],
        ref_freq=1e8,
        spectral_index=-0.5,
        peak_normalized=True,
    )
    print(f"\n4. ZernikeBeam")
    print(f"   ref_freq: {zernike.ref_freq/1e6:.0f} MHz")
    print(f"   spectral_index: {zernike.spectral_index}")
    print(f"   peak_normalized: {zernike.peak_normalized}")

<a id="section-3"></a>
## Section 3: CLI Options Reference

### Command: `runsim` (Main Simulation)

```bash
python vsim.py runsim [OPTIONS]
```

### Analytic Beam Options

| Option | Type | Default | Description |
|--------|------|---------|-------------|
| `--analytic-beam-class` | choice | None | Beam class to use. Choices: `AiryBeam`, `GaussianBeam`, `hera_sim.beams.PolyBeam`, `hera_sim.beams.ZernikeBeam`, `hera_sim.beams.PerturbedPolyBeam` |
| `--analytic-beam-diameter` | float | 14.0 | Dish diameter in meters (for AiryBeam) |
| `--analytic-beam-sigma` | float | 0.15 | Beam width in radians (for GaussianBeam) |
| `--analytic-beam-ref-freq` | float | 1.0e8 | Reference frequency in Hz |
| `--analytic-beam-spectral-index` | float | -0.6975 | Frequency scaling power law index |
| `--analytic-beam-coeffs-file` | path | None | JSON/YAML file with beam coefficients |
| `--analytic-beam-preset` | choice | None | Preset coefficients. Choices: `fagnoni19` |
| `--analytic-beam-map-file` | path | None | YAML file mapping antennas to beam configs |

### Per-Antenna UVBeam Options

| Option | Type | Default | Description |
|--------|------|---------|-------------|
| `--beam-map-csv` | path | None | CSV with columns `ant_number,beam_file`. **matvis only** |
| `--beamvar-type` | choice | None | Beam variation preset: `vivaldired`, `airyred`, `airyprb`, `airytilt` |

### Simulator Options

| Option | Type | Default | Description |
|--------|------|---------|-------------|
| `--simulator` | choice | `matvis` | Backend: `matvis`, `matvis-cpu`, `fftvis`, `fftvis32`, `fftvis64` |

---

### Option Interactions

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  OPTION PRECEDENCE & INTERACTIONS                                           │
└─────────────────────────────────────────────────────────────────────────────┘

IF --analytic-beam-map-file is set:
    → Ignores --beam-map-csv, --beamvar-type
    → Uses YAML-defined per-antenna analytic beams
    
ELIF --analytic-beam-class is set:
    → Ignores --beam-map-csv, --beamvar-type  
    → Uses single analytic beam for all antennas
    → Requires appropriate params:
        • AiryBeam: --analytic-beam-diameter
        • GaussianBeam: --analytic-beam-sigma (or diameter)
        • PolyBeam/ZernikeBeam: --analytic-beam-preset OR --analytic-beam-coeffs-file
    
ELIF --beam-map-csv is set AND --simulator starts with "matvis":
    → Per-antenna UVBeam files from CSV
    
ELSE:
    → Default beam: beams/NF_HERA_Vivaldi_efield_beam_extrap.fits
```

<a id="section-4"></a>
## Section 4: Configuration File Formats

### 4.1 Analytic Beam Map YAML (`--analytic-beam-map-file`)

This YAML format allows specifying **different analytic beams per antenna**.

**File Location Example:** `beams/analytic_beam_map.yaml`

```yaml
# Shared defaults for all beams
defaults:
  class: hera_sim.beams.ZernikeBeam
  ref_freq: 100000000.0    # 100 MHz
  spectral_index: -0.5
  peak_normalized: true

# Unique beam definitions (beam_id → params)
beam_definitions:
  0:  # nominal symmetric beam
    beam_coeffs: [1.0, 0.0, 0.0, 0.0, -0.2, 0.0, 0.0, 0.0, 0.0, 0.05]
  1:  # X-tilted beam
    beam_coeffs: [1.0, 0.03, 0.0, 0.0, -0.2, 0.0, 0.0, 0.0, 0.0, 0.05]
  2:  # Y-tilted beam  
    beam_coeffs: [1.0, 0.0, 0.03, 0.0, -0.2, 0.0, 0.0, 0.0, 0.0, 0.05]
  3:  # diagonal tilt
    beam_coeffs: [1.0, 0.02, 0.02, 0.0, -0.2, 0.0, 0.0, 0.0, 0.0, 0.05]

# Antenna to beam_id mapping
antenna_mapping:
  0: 0    # Antenna 0 uses beam 0
  1: 0    # Antenna 1 uses beam 0
  2: 1    # Antenna 2 uses beam 1 (X-tilted)
  3: 0
  4: 2    # Antenna 4 uses beam 2 (Y-tilted)
  5: 3
  6: 0
  7: 3
  8: 1
  9: 0

default_beam_id: 0  # For antennas not listed in antenna_mapping
```

**Key Structure:**
- `defaults`: Parameters shared by all beams (merged with per-beam overrides)
- `beam_definitions`: Dict of beam_id → beam parameters
- `antenna_mapping`: Dict of antenna_number → beam_id
- `default_beam_id`: Fallback beam for unlisted antennas

---

### 4.2 Beam Coefficients File (`--analytic-beam-coeffs-file`)

For PolyBeam/ZernikeBeam, coefficients can be loaded from a file.

**YAML Format:**
```yaml
beam_coeffs:
- 1.0
- 0.03
- 0.01
- 0.0
- -0.25
- 0.0
```

**JSON Format:**
```json
{
  "beam_coeffs": [1.0, 0.03, 0.01, 0.0, -0.25, 0.0]
}
```

---

### 4.3 Per-Antenna UVBeam CSV (`--beam-map-csv`)

Maps each antenna to a UVBeam FITS file.

**Format:**
```csv
ant_number,beam_file
0,/path/to/beam_0.fits
1,/path/to/beam_0.fits
2,/path/to/beam_1.fits
3,/path/to/beam_0.fits
```

**Note:** Only works with `--simulator matvis` or `--simulator matvis-cpu`.

In [ ]:
# ============================================================
# Inspect Existing Configuration Files
# ============================================================

from pathlib import Path
import yaml

VSIM_ROOT = Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim")
BEAMS_DIR = VSIM_ROOT / "beams"

print("Available Configuration Files:")
print("=" * 60)

# Check for YAML beam maps
yaml_files = list(BEAMS_DIR.glob("*.yaml"))
print(f"\nYAML files in beams/: {[f.name for f in yaml_files]}")

# Check for CSV beam maps
csv_files = list(VSIM_ROOT.glob("beam_map*.csv"))
print(f"\nCSV beam maps in root: {[f.name for f in csv_files]}")

# Read and display the analytic_beam_map.yaml
abeam_map = BEAMS_DIR / "analytic_beam_map.yaml"
if abeam_map.exists():
    print(f"\n{'='*60}")
    print(f"Contents of {abeam_map.name}:")
    print("=" * 60)
    with open(abeam_map) as f:
        content = f.read()
    print(content)

# Read and display zernike_tilted.yaml
ztilt = BEAMS_DIR / "zernike_tilted.yaml"
if ztilt.exists():
    print(f"\n{'='*60}")
    print(f"Contents of {ztilt.name}:")
    print("=" * 60)
    with open(ztilt) as f:
        content = f.read()
    print(content)

<a id="section-5"></a>
## Section 5: Code Architecture & Call Flow

### Module Structure

```
validation-sim/
├── vsim.py                         # CLI entry point (Click framework)
│   ├── runsim()                    # Main simulation command
│   └── make-obsparams()            # Generate observation parameters
│
└── core/
    ├── anabeam_config.py           # Analytic beam config builders
    │   ├── read_analytic_beam_map()    # Parse YAML beam map
    │   └── build_analytic_beam_config() # Build config dict from CLI args
    │
    ├── obsparams.py                # Observation parameter generation
    │   ├── make_hera_obsparam()    # Create full obsparam directory
    │   ├── make_layout_with_beamids() # Add BeamID column to layout
    │   └── make_tele_config()      # Generate telescope config YAML
    │
    ├── run_sim.py                  # SLURM job submission
    │   └── run_validation_sim()    # Main simulation runner
    │
    └── slurm.py                    # SLURM utilities
        └── @slurmify decorator
```

### Call Flow Diagram

```
┌────────────────────────────────────────────────────────────────────────────┐
│ USER COMMAND:                                                               │
│   python vsim.py runsim --ants 0~10 --channels 227~257                     │
│       --analytic-beam-map-file beams/analytic_beam_map.yaml                │
└────────┬───────────────────────────────────────────────────────────────────┘
         │
         ▼
┌────────────────────────────────────────────────────────────────────────────┐
│ vsim.py :: runsim()                                                         │
│   │                                                                          │
│   ├── Parse CLI options                                                     │
│   │                                                                          │
│   ├── IF --analytic-beam-map-file:                                          │
│   │       analytic_beam_map_file_path = Path(...)                           │
│   │       beam_map_csv = None  # Override UVBeam CSV                        │
│   │                                                                          │
│   ├── ELIF --analytic-beam-class:                                           │
│   │       analytic_beam = build_analytic_beam_config(...)                   │
│   │           └── core/anabeam_config.py                                    │
│   │                                                                          │
│   └── run_validation_sim(                                                   │
│           analytic_beam=...,                                                │
│           analytic_beam_map_file=...,                                       │
│       )                                                                      │
└────────┬───────────────────────────────────────────────────────────────────┘
         │
         ▼
┌────────────────────────────────────────────────────────────────────────────┐
│ core/run_sim.py :: run_validation_sim()                                     │
│   │                                                                          │
│   ├── layout_file = make_hera_obsparam(                                     │
│   │       analytic_beam=analytic_beam,                                      │
│   │       analytic_beam_map_file=analytic_beam_map_file,                    │
│   │   )                                                                      │
│   │   └── core/obsparams.py                                                 │
│   │                                                                          │
│   └── Submit SLURM job with hera-sim-vis.py                                 │
└────────┬───────────────────────────────────────────────────────────────────┘
         │
         ▼
┌────────────────────────────────────────────────────────────────────────────┐
│ core/obsparams.py :: make_hera_obsparam()                                   │
│   │                                                                          │
│   ├── IF analytic_beam_map_file:                                            │
│   │       beam_defs, ant_to_beamid = read_analytic_beam_map(file)           │
│   │           └── core/anabeam_config.py                                    │
│   │       analytic_beam_map = convert_to_tuple(beam_defs)                   │
│   │                                                                          │
│   ├── layout_file = make_layout_with_beamids(                               │
│   │       analytic_beam_map=analytic_beam_map,                              │
│   │   )                                                                      │
│   │   # Adds BeamID column to antenna layout file                           │
│   │                                                                          │
│   └── tele_config = make_tele_config(                                       │
│           analytic_beam_map=analytic_beam_map,                              │
│       )                                                                      │
│       # Generates telescope config with !AnalyticBeam tags                  │
└────────┬───────────────────────────────────────────────────────────────────┘
         │
         ▼
┌────────────────────────────────────────────────────────────────────────────┐
│ OUTPUT: config_files/obsparams/<model>/                                     │
│   ├── obsparam.yaml              # Full observation parameters             │
│   ├── layout_with_beamids.txt    # Antenna positions + BeamID column       │
│   └── teleconfig.yaml            # Telescope config with beam definitions  │
│                                                                              │
│ teleconfig.yaml example:                                                     │
│   beam_paths:                                                                │
│     0: !AnalyticBeam                                                        │
│       class: hera_sim.beams.ZernikeBeam                                     │
│       beam_coeffs: [1.0, 0.0, 0.0, 0.0, -0.2, ...]                         │
│       ref_freq: 100000000.0                                                 │
│     1: !AnalyticBeam                                                        │
│       class: hera_sim.beams.ZernikeBeam                                     │
│       beam_coeffs: [1.0, 0.03, 0.0, 0.0, -0.2, ...]                        │
└────────────────────────────────────────────────────────────────────────────┘
```

<a id="section-6"></a>
## Section 6: Example Commands

### 6.1 Single Analytic Beam (All Antennas Same)

#### AiryBeam (14m dish)
```bash
python vsim.py runsim \
    --ants 0~10 \
    --channels 227~257 \
    --sky-model ptsrc \
    --simulator matvis \
    --analytic-beam-class AiryBeam \
    --analytic-beam-diameter 14.0
```

#### GaussianBeam
```bash
python vsim.py runsim \
    --ants 0~10 \
    --channels 227~257 \
    --sky-model ptsrc \
    --simulator matvis \
    --analytic-beam-class GaussianBeam \
    --analytic-beam-sigma 0.15
```

#### PolyBeam with Fagnoni19 preset
```bash
python vsim.py runsim \
    --ants 0~10 \
    --channels 227~257 \
    --sky-model ptsrc \
    --simulator matvis \
    --analytic-beam-class hera_sim.beams.PolyBeam \
    --analytic-beam-preset fagnoni19
```

#### ZernikeBeam with custom coefficients file
```bash
python vsim.py runsim \
    --ants 0~10 \
    --channels 227~257 \
    --sky-model ptsrc \
    --simulator matvis \
    --analytic-beam-class hera_sim.beams.ZernikeBeam \
    --analytic-beam-coeffs-file beams/zernike_tilted.yaml \
    --analytic-beam-ref-freq 1.0e8 \
    --analytic-beam-spectral-index -0.5
```

---

### 6.2 Multi-Antenna Analytic Beam Map

```bash
python vsim.py runsim \
    --ants 0~10 \
    --channels 227~257 \
    --sky-model ptsrc \
    --simulator matvis \
    --analytic-beam-map-file beams/analytic_beam_map.yaml
```

---

### 6.3 Per-Antenna UVBeam Files (matvis only)

```bash
python vsim.py runsim \
    --ants 0~10 \
    --channels 227~257 \
    --sky-model ptsrc \
    --simulator matvis \
    --beam-map-csv beam_map_10_airy.csv
```

---

### 6.4 With EOR-GRF Sky Model

```bash
python vsim.py runsim \
    --ants 0~50 \
    --channels 227~316 \
    --sky-model eor-grf-256 \
    --n-time-chunks 288 \
    --do-time-chunks 0~10 \
    --simulator matvis \
    --analytic-beam-class hera_sim.beams.PolyBeam \
    --analytic-beam-preset fagnoni19
```

---

### 6.5 Generate Only Obsparams (No Simulation)

```bash
python vsim.py make-obsparams \
    --ants 0~10 \
    --channels 227~257 \
    --sky-model ptsrc \
    --analytic-beam-class hera_sim.beams.ZernikeBeam \
    --analytic-beam-coeffs-file beams/zernike_tilted.yaml
```

<a id="section-7"></a>
## Section 7: Output & Log File Locations

### Directory Structure

```
validation-sim/
├── beams/                          # Beam files and configs
│   ├── analytic_beam_map.yaml      # Multi-antenna analytic beam config
│   ├── zernike_tilted.yaml         # Zernike coefficients file
│   ├── *.fits                      # UVBeam FITS files
│   └── *.h5                        # HDF5 beam files
│
├── beam_map_*.csv                  # Per-antenna UVBeam CSV maps
│
├── config_files/
│   └── obsparams/                  # Generated observation parameters
│       └── <sky_model>/
│           └── <layout>/
│               ├── obsparam_*.yaml
│               ├── layout_*.with_beamids
│               └── teleconfig_*.yaml
│
├── outputs/                        # Simulation output visibilities
│   └── <sky_model>/
│       └── <layout>/
│           └── *.uvh5
│
├── logs/
│   └── vis/                        # Visibility simulation logs
│       └── <sky_model>/
│           └── <layout>/
│               └── *-<JOBID>.out
│
└── batch_scripts/
    └── vis/                        # Generated SLURM scripts
        └── *.sbatch
```

### Finding Your Logs

```bash
# Check running jobs
squeue -u $USER

# Find most recent visibility logs
ls -lt logs/vis/<sky_model>/<layout>/*.out | head -5

# Tail a specific job log
tail -f logs/vis/<sky_model>/<layout>/<jobname>-<JOBID>.out

# Find generated obsparams
ls config_files/obsparams/<sky_model>/<layout>/

# View generated telescope config
cat config_files/obsparams/<sky_model>/<layout>/teleconfig_*.yaml
```

In [ ]:
# ============================================================
# List Existing Beam Files and Configs
# ============================================================

from pathlib import Path

VSIM_ROOT = Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim")

print("Beam Files and Configurations:")
print("=" * 60)

# Beams directory
beams_dir = VSIM_ROOT / "beams"
if beams_dir.exists():
    print(f"\n{beams_dir}/")
    for f in sorted(beams_dir.iterdir()):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:50s} ({size_kb:8.1f} KB)")

# CSV beam maps
print(f"\n{VSIM_ROOT}/")
for f in sorted(VSIM_ROOT.glob("beam_map*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:50s} ({size_kb:8.1f} KB)")

# Sample obsparams with beam configs
obsparams_dir = VSIM_ROOT / "config_files" / "obsparams"
if obsparams_dir.exists():
    print(f"\nSample teleconfig files:")
    for tele in list(obsparams_dir.rglob("teleconfig*.yaml"))[:3]:
        rel_path = tele.relative_to(VSIM_ROOT)
        print(f"  {rel_path}")

<a id="section-8"></a>
## Section 8: Source Code Excerpts

### 8.1 `core/anabeam_config.py` — Beam Config Builders

This module provides functions to:
1. Read YAML beam map files (`read_analytic_beam_map`)
2. Build beam config dicts from CLI arguments (`build_analytic_beam_config`)

In [ ]:
# ============================================================
# Show source code from anabeam_config.py
# ============================================================

from pathlib import Path

anabeam_file = Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/core/anabeam_config.py")

if anabeam_file.exists():
    print(f"# File: {anabeam_file.name}")
    print("=" * 70)
    with open(anabeam_file) as f:
        content = f.read()
    print(content)

### 8.2 `core/obsparams.py` — Telescope Config Generation

The `make_tele_config()` function generates the telescope configuration YAML, which includes the beam definitions.

In [ ]:
# ============================================================
# Show relevant section from obsparams.py
# ============================================================

from pathlib import Path

obsparams_file = Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/core/obsparams.py")

if obsparams_file.exists():
    with open(obsparams_file) as f:
        lines = f.readlines()
    
    # Find make_tele_config function
    print("# File: obsparams.py — make_tele_config() function")
    print("=" * 70)
    
    in_func = False
    func_lines = []
    for i, line in enumerate(lines, 1):
        if "def make_tele_config" in line:
            in_func = True
        if in_func:
            func_lines.append(f"{i:4d} | {line.rstrip()}")
            # Stop after ~100 lines of the function
            if len(func_lines) > 100:
                func_lines.append("     | ... (truncated)")
                break
    
    print("\n".join(func_lines))

### 8.3 `vsim.py` — CLI Option Definitions

The `runsim` command in vsim.py defines all the analytic beam CLI options.

In [ ]:
# ============================================================
# Show CLI options from vsim.py
# ============================================================

from pathlib import Path

vsim_file = Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/vsim.py")

if vsim_file.exists():
    with open(vsim_file) as f:
        lines = f.readlines()
    
    print("# File: vsim.py — Analytic beam CLI options (lines 35-95)")
    print("=" * 70)
    
    for i, line in enumerate(lines[34:95], start=35):
        print(f"{i:4d} | {line.rstrip()}")

## Summary: Quick Reference

### Beam Mode Selection

| Want to... | Use This |
|------------|----------|
| Same analytic beam for all antennas | `--analytic-beam-class` + params |
| Different analytic beams per antenna | `--analytic-beam-map-file` |
| Different UVBeam files per antenna | `--beam-map-csv` (matvis only) |
| Default Vivaldi beam | (no beam options) |

### Beam Class Selection

| Beam Type | Class Name | Key Parameters |
|-----------|-----------|----------------|
| Airy disk | `AiryBeam` | `diameter` |
| Gaussian | `GaussianBeam` | `sigma` or `diameter` |
| Chebyshev polynomial | `hera_sim.beams.PolyBeam` | `beam_coeffs`, `ref_freq`, `spectral_index` |
| Zernike polynomial | `hera_sim.beams.ZernikeBeam` | `beam_coeffs`, `ref_freq`, `spectral_index`, `peak_normalized` |
| Perturbed polynomial | `hera_sim.beams.PerturbedPolyBeam` | PolyBeam + perturbation params |

### File Locations

| File Type | Location |
|-----------|----------|
| Analytic beam map YAML | `beams/analytic_beam_map.yaml` |
| Zernike coefficients | `beams/zernike_tilted.yaml` |
| UVBeam CSV maps | `beam_map_*.csv` |
| UVBeam FITS files | `beams/*.fits` |
| Generated teleconfig | `config_files/obsparams/<model>/<layout>/teleconfig_*.yaml` |
| Simulation logs | `logs/vis/<model>/<layout>/*.out` |